In [1]:
from pathlib import Path
from dataclasses import asdict, replace
import json
from dataset import device_for_run, load_store, prepare_reward_cache, reward_splits
from reward_model import RewardSettings, train_reward, evaluate_reward
from tracking import start_task


In [2]:
ROOT = Path.cwd()
DATA = ROOT / 'data' / 'Yambda'
VOCAB = ROOT / 'checkpoints' / 'gpt2' / 'vocab.json'
CACHE = ROOT / 'results' / 'reward_events.parquet'
LOG_CLEARML = True
SEEDS = (42, 43, 44)
settings = RewardSettings(embedding_dim=64, hidden_dim=128, patience=3, epochs=10, batch_size=256, learning_rate=1e-3, weight_decay=0.01, gradient_clip=1.0, max_train_rows=150000, max_eval_rows=20000, duration_scale_seconds=600.0, like_weight=0.3)
device = device_for_run()
gpt_settings = json.loads((ROOT / 'checkpoints' / 'gpt2' / 'settings.json').read_text())
store = load_store(DATA, max_length=gpt_settings['max_length'], vocab_path=VOCAB)
cache = prepare_reward_cache(DATA, CACHE)
splits = reward_splits(cache, store.info)


In [3]:
all_metrics = {}
for seed in SEEDS:
    run_settings = replace(settings, seed=seed)
    checkpoint = ROOT / 'checkpoints' / f'reward_seed_{seed}'
    task = start_task(f'reward seed {seed}', {'settings': asdict(run_settings), 'catalog_items': len(store.catalog.items)}, LOG_CLEARML)
    model, history = train_reward(store, splits, run_settings, checkpoint, device, task)
    all_metrics[str(seed)] = {'validation': evaluate_reward(model, splits['validation'], store, run_settings, device), 'test': evaluate_reward(model, splits['test'], store, run_settings, device)}
    if task:
        for split, metrics in all_metrics[str(seed)].items():
            for name, value in metrics.items():
                task.get_logger().report_single_value(f'{split}/{name}', value)
        task.close()
    print(seed, all_metrics[str(seed)])
(ROOT / 'results' / 'reward_metrics.json').write_text(json.dumps(all_metrics, indent=2))


ClearML Task: created new task id=c85fe284b69c4815b682617be7ceb57d
ClearML results page: https://app.clear.ml/projects/b41b8f398f364b9cbe1b273c663b6cea/tasks/c85fe284b69c4815b682617be7ceb57d/output/log
2026-10-25 16:10:06,507 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
Early stopping at epoch 7. Best validation ratio RMSE: 0.4184
42 {'validation': {'events': 20000, 'ratio_mae': 0.35845011472702026, 'ratio_rmse': 0.41842193634398567, 'ratio_spearman': 0.3612039064658864, 'duration_mae': 0.12478023767471313, 'duration_rmse': 0.15149539317233718, 'duration_seconds_mae': 74.86814260482788, 'duration_seconds_rmse': 90.8972359034023, 'duration_spearman': 0.30709581007017334, 'like_roc_auc': 0.6775583783009408, 'like_pr_auc': 0.8901232592553406, 'like_examples': 1805}, 'test': {'events': 20000, 'ratio_mae': 0.3733104169368744, 'ratio_rmse': 0.43008887877

Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/b41b8f398f364b9cbe1b273c663b6cea/tasks/cb153b8e434242e3919827795e4520b3/output/log
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
Early stopping at epoch 7. Best validation ratio RMSE: 0.4277


Connecting multiple input models with the same name: `best`. This might result in the wrong model being used when executing remotely


43 {'validation': {'events': 20000, 'ratio_mae': 0.36601969599723816, 'ratio_rmse': 0.4277443383854934, 'ratio_spearman': 0.31972453318668037, 'duration_mae': 0.12627090513706207, 'duration_rmse': 0.15280215669119152, 'duration_seconds_mae': 75.76254308223724, 'duration_seconds_rmse': 91.68129401471491, 'duration_spearman': 0.30002091031469785, 'like_roc_auc': 0.6998611177435015, 'like_pr_auc': 0.9003403343375453, 'like_examples': 1805}, 'test': {'events': 20000, 'ratio_mae': 0.37288030982017517, 'ratio_rmse': 0.428979237917694, 'ratio_spearman': 0.2953111600253004, 'duration_mae': 0.13034990429878235, 'duration_rmse': 0.15601162608775807, 'duration_seconds_mae': 78.20994257926941, 'duration_seconds_rmse': 93.60697565265484, 'duration_spearman': 0.25691515858533165, 'like_roc_auc': 0.6549023100156556, 'like_pr_auc': 0.8483158638488564, 'like_examples': 1919}}
ClearML Task: created new task id=3d27b09c1f5d485f92474ea9f263f45f


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/b41b8f398f364b9cbe1b273c663b6cea/tasks/3d27b09c1f5d485f92474ea9f263f45f/output/log
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
Early stopping at epoch 6. Best validation ratio RMSE: 0.4205
44 {'validation': {'events': 20000, 'ratio_mae': 0.36129122972488403, 'ratio_rmse': 0.4204970036387423, 'ratio_spearman': 0.34913889826729355, 'duration_mae': 0.12528245151042938, 'duration_rmse': 0.15153255043840622, 'duration_seconds_mae': 75.16947090625763, 'duration_seconds_rmse': 90.91953026304373, 'duration_spearman': 0.3085096642809186, 'like_roc_auc': 0.6952370151059325, 'like_pr_auc': 0.9011509252917529, 'like_examples': 1805}, 'test': {'events': 20000, 'ratio_mae': 0.37037959694862366, 'ratio_rmse': 0.425349287646549, 'ratio_spearman': 0.311014658662753, 'duration_mae': 0.1290930211544037, 'duration_rmse': 0.1549758218165941, 'duration_seconds_mae': 77.45581269264221, 'duration_seconds_rmse': 92

3135

In [4]:
import numpy as np
from dataset import reward_splits
from reward_model import RewardDataset

splits = reward_splits(ROOT / "results" / "reward_events.parquet", store.info)

train = RewardDataset(
    splits["train"], store, settings,
    settings.max_train_rows, seed=42
)

val = RewardDataset(
    splits["validation"], store, settings,
    settings.max_eval_rows, seed=42
)

train_ratio = train.targets[0][train.targets[3]]
val_ratio = val.targets[0][val.targets[3]]

mean_ratio = train_ratio.mean()

mae = np.mean(np.abs(val_ratio - mean_ratio))
rmse = np.sqrt(np.mean((val_ratio - mean_ratio) ** 2))

print(f"Constant baseline MAE:  {mae:.4f}")
print(f"Constant baseline RMSE: {rmse:.4f}")
print()
print("Reward model MAE:       0.3585")
print("Reward model RMSE:      0.4184")

Constant baseline MAE:  0.4231
Constant baseline RMSE: 0.4456

Reward model MAE:       0.3585
Reward model RMSE:      0.4184
